#### N8 — kNN Text Peer Lists (M2)

Builds M2 peer lists via cosine kNN in FinBERT 768-dim embedding space.
Since embeddings are L2-normalised, dot product = cosine similarity.

**Input:** `text_embeddings.parquet` (N7)
**Output:** `peers_m2.parquet`


In [ ]:
# imports & config
import sys
from pathlib import Path

notebook_dir = Path('__file__').parent if '__file__' in dir() else Path.cwd()
repo_root = next(
    (p for p in [notebook_dir, *notebook_dir.parents] if (p / 'config.py').exists()),
    None
)
if repo_root is None:
    raise FileNotFoundError('config.py not found — check repo structure')
sys.path.insert(0, str(repo_root))

from config import *
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

print('Config loaded.')
print(f'  EMBEDDINGS : {EMBEDDINGS}')
print(f'  PEERS_M2   : {PEERS_M2}')
print(f'  K_MAIN     : {K_MAIN}')


In [ ]:
# declare I/O
INPUTS  = [EMBEDDINGS]
OUTPUTS = [PEERS_M2]
# PURPOSE : Build M2 peer lists via cosine kNN in FinBERT embedding space
# RUNTIME : ~15 min
# DEPENDS : text_embeddings.parquet (N7)


### 1 · Load Embeddings


In [ ]:
# load embeddings
df_emb = pd.read_parquet(EMBEDDINGS)
emb_cols = [f'emb_{i}' for i in range(EMBEDDING_DIM)]

print(f"Loaded: {df_emb.shape[0]:,} firm-years × {df_emb.shape[1]} columns")
print(f"Embedding dimensions: {len(emb_cols)}")
print(f"Years: {sorted(df_emb['fyear'].unique())}")
print()
print("Coverage per year:")
print(df_emb.groupby('fyear')['tic'].count().to_string())


### 2 · Cosine kNN — Per Fyear


In [ ]:
def build_peer_list(df_year, emb_cols, fyear, k_max, model_name='M2_Text'):
    tickers    = df_year['tic'].values
    E          = df_year[emb_cols].values   # already L2-normalised
    sim_matrix = cosine_similarity(E)       # dot product = cosine for L2-normed
    np.fill_diagonal(sim_matrix, -np.inf)

    records = []
    k = min(k_max, len(tickers) - 1)

    for i in range(len(tickers)):
        top_k_idx = np.argpartition(sim_matrix[i], -k)[-k:]
        top_k_idx = top_k_idx[np.argsort(sim_matrix[i][top_k_idx])[::-1]]
        for rank, j in enumerate(top_k_idx, start=1):
            records.append({
                'focal_tic'       : tickers[i],
                'focal_fyear'     : fyear,
                'peer_tic'        : tickers[j],
                'rank'            : rank,
                'similarity_score': float(sim_matrix[i][j]),
                'model'           : model_name,
            })
    return pd.DataFrame(records)

print('build_peer_list() defined.')


In [ ]:
# run kNN for all years
K_MAX    = max(K_ROBUSTNESS)
all_peers = []

print(f"Building M2 peer lists (cosine kNN, k_max={K_MAX})...")
print()

for yr in YEARS:
    df_yr = df_emb[df_emb['fyear'] == yr].dropna(subset=['tic']).reset_index(drop=True)
    if len(df_yr) == 0:
        print(f"  {yr}: no embeddings — skipping")
        continue

    peers_yr = build_peer_list(df_yr, emb_cols, yr, K_MAX)
    all_peers.append(peers_yr)

    avg_sim = peers_yr[peers_yr['rank'] == 1]['similarity_score'].mean()
    print(f"  {yr}: {len(df_yr):,} firms → {len(peers_yr):,} records | "
          f"avg top-1 similarity: {avg_sim:.4f}")

peers_m2 = pd.concat(all_peers, ignore_index=True)
print(f"\nTotal: {len(peers_m2):,} peer records")


In [ ]:
# schema validation
print("Schema check:")
for col in PEER_SCHEMA:
    present = col in peers_m2.columns
    print(f"  {col:<20} {'✓' if present else '✗'}  {peers_m2[col].dtype if present else 'MISSING'}")

print()
print("Null check:")
print(peers_m2.isnull().sum().to_string())


In [ ]:
df_panel = pd.read_parquet(PANEL_CLEAN, columns=['tic', 'fyear', 'ff49_num'])

peers_check = peers_m2[peers_m2['rank'] <= K_MAIN].copy()
peers_check = peers_check.merge(
    df_panel.rename(columns={'tic': 'focal_tic', 'fyear': 'focal_fyear',
                              'ff49_num': 'focal_ff49'}),
    on=['focal_tic', 'focal_fyear'], how='left'
)
peers_check = peers_check.merge(
    df_panel.rename(columns={'tic': 'peer_tic', 'fyear': 'focal_fyear',
                              'ff49_num': 'peer_ff49'}),
    on=['peer_tic', 'focal_fyear'], how='left'
)
peers_check['same_ff49'] = peers_check['focal_ff49'] == peers_check['peer_ff49']
same_rate = peers_check['same_ff49'].mean() * 100

print(f'M2 Text kNN — FF49 same-industry rate (k={K_MAIN}): {same_rate:.1f}%')
print(f'  Crosses FF49 boundaries: {100-same_rate:.1f}% of top-{K_MAIN} peers')


In [ ]:
peers_m2.to_parquet(PEERS_M2, index=False)

print(f'Saved : {PEERS_M2}')
print(f'Shape : {peers_m2.shape[0]:,} rows × {peers_m2.shape[1]} columns')
print()
print('Peer records per year:')
print(peers_m2.groupby('focal_fyear')['focal_tic'].count().to_string())
print()
print('Next: N9 — Late Fusion (M3)')
